# AutoRound Quantization Pipeline
W4 + Layer Protection Strategy

In [ ]:
# =========================
# Import
# =========================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import torch
import shutil
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from auto_round import AutoRound

In [ ]:
# =========================
# Setting
# =========================
MODEL_ID = "/workspace/base_model"
OUT_DIR  = "/workspace/jw/model"
NUM_CALIBRATION_SAMPLES = 1024

In [ ]:
# =========================
# Model Load
# =========================
print("[INFO] 모델 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print("[INFO] 모델 로드 완료")

In [ ]:
# =========================
# Dataset Load
# =========================
print("[INFO] 캘리브레이션 데이터 로드 중...")
ds = load_dataset(
    "LGAI-EXAONE/MANTA-1M",
    split=f"train[:{NUM_CALIBRATION_SAMPLES}]"
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)
texts = [x["text"] for x in ds]

# 길이 필터
texts = [t for t in texts if len(t) > 200]
print(f"[INFO] 사용 데이터 수: {len(texts)}")

In [ ]:
# =========================
# 레이어 보호 (핵심)
# =========================
def get_quantizable_layers(model):
    layers = []
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            # 보호할 레이어 제외
            if any(skip in name for skip in [
                "lm_head",
                "embed_tokens",
                "self_attn.q_proj",
                "self_attn.k_proj"
            ]):
                continue
            layers.append(name)
    return layers

quant_layers = get_quantizable_layers(model)
print(f"[INFO] 양자화 대상 레이어 수: {len(quant_layers)}")

In [ ]:
# =========================
# Quantization
# =========================
print("[INFO] AutoRound 시작...")
autoround = AutoRound(
    model=model,
    tokenizer=tokenizer,
    bits=4,           # W4 유지
    group_size=128,   # 최적값
    sym=False,
    dataset=texts,
    nsamples=1024,
    seqlen=512,
    iters=200,
    quant_layers=quant_layers  # 핵심: 보호 레이어 제외
)
autoround.quantize()
print("[INFO] 양자화 완료")

In [ ]:
# =========================
# Save
# =========================
print("[INFO] 모델 저장 중...")
os.makedirs(OUT_DIR, exist_ok=True)
autoround.save_quantized(
    OUT_DIR,
    format="awq",
    inplace=True,
)
tokenizer.save_pretrained(OUT_DIR)
print(f"[INFO] 저장 완료: {OUT_DIR}")

In [ ]:
# =========================
# Zip
# =========================
shutil.make_archive(
    base_name="/workspace/jw/autoround_submit",
    format="zip",
    root_dir="/workspace/jw",
    base_dir="model"
)
print("[INFO] zip 완료")

In [ ]:
# =========================
# Submit
# =========================
from dacon_submit_api import dacon_submit_api

result = dacon_submit_api.post_code_submission_file(
    '/workspace/jw/autoround_submit.zip',
    '토큰입력',
    '236689',
    '야망guys',
    'W4 + layer protection final'
)
print(result)